In [ ]:
# 자동차 자율 주행
# 1차원 공간에서 위치를 이동하며 중앙에 머무는 것이 목표
# 환경은 1차선 도로, 에이전트(차량)는 좌우에 치우치지 않게 중앙으로 유지
import numpy as np
import random
import matplotlib.pyplot as plt

In [ ]:
state_space = np.linspace(-1.0, 1.0, 11)    # 공간을 11개로 나눔
print(state_space)
action_space = [-1, 0, 1]    # 좌 중간 우 ; 에이전트가 할 수 있는 행동 3가지

q_table = np.zeros((len(state_space), len(action_space)))    # 11 * 3의 0으로 채워진 2차원 배열
# print(q_table)

# 학습 하이퍼 파라미터
alpha = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 500

def get_state_index(position):    # 연속적인 값을 이산화
  return np.argmin(np.abs(state_space - position))    # argmin / argmax : 최소/최대값이 어디에 있는지 인덱스 값 반환

# print(get_state_index(-0.1))
# print(get_state_index(0.5))
 
# 보상 : 0에 가까워지면 보상 커진다. 
def get_reward(position):
  return -abs(position)

# 환경의 동작 정의
def stepFunc(position, action):
  position += action * 0.1    # 현재 위치에 행동(action_state)을 반영
  position = np.clip(position, -1.0, 1.0)    # -1.0 < position < 1.0 ; np.clip(a, a_min, a_max)
  reward = get_reward(position)
  return position, reward

reward_list=[]

# agent의 학습 루프 : 행동을 선택하고 학습을 반복
for ep in range(episodes):
  position = np.random.uniform(-1.0, 1.0)    # 초기 위치는 랜덤
  total_reward = 0

  for _ in range(50):     # 한 개의 에피소드마다 50번의 action을 반복
    state_idx = get_state_index(position)

    if random.random() < epsilon:
      action_idx = random.choice([0, 1, 2])
    else:
      action_idx = np.argmax(q_table[state_idx])

    # 선택된 행동을 환경에 적용
    action = action_space[action_idx]   # 행동 선택
    next_position, reward = stepFunc(position, action)    # 환경에 적용
    next_state_idx = get_state_index(next_position)    # 다음 위치 인덱스
    
    best_next_q = np.max(q_table[next_state_idx])

    # q_table 갱신(벨만 방정식)
    q_table[state_idx, action_idx] += alpha * (reward + gamma * best_next_q - q_table[state_idx, action_idx])
    
    position = next_position
    total_reward += reward

  reward_list.append(total_reward)

  if ep % 50 == 0:    # 50 episode마다
    initial_avg = np.mean(reward_list[:50])
    final_avg = np.mean(reward_list[-50:])
    max_reward = np.max(reward_list)
    min_reward = np.min(reward_list)
    print("Performance summary")
    print(f"- inital 50 episodes avg reward : {initial_avg:.3f}")
    print(f"- final 50 episodes avg reward : {final_avg:.3f}")
    print(f"- max reward : {max_reward:.3f}")
    print(f"- min reward : {min_reward:.3f}")

    # 보상 향상 여부 확인
    if final_avg > initial_avg:
      print(f"모델이 걔선됨. (+){final_avg - initial_avg:.3f}")
    else:
      print(f"모델이 걔선되지 않음. 파라미터 조정이 필요 (-){initial_avg - final_avg:.3f}")


In [ ]:
# 보상 변화 시각화
plt.figure(figsize=(10,5))
plt.plot(reward_list, label="episode reward")
plt.axhline(y=0, color='gray', linestyle='--', lw=1)
plt.xlabel("episode")
plt.ylabel("reward")
plt.grid(True)
plt.legend()
plt.show()
plt.close()

In [ ]:
# 에피소드 50개 단위로 평균 보상 시각화
window = 50
avg_rewards = []

for i in range(0, len(reward_list), window):
  chunk = reward_list[i:i+window]
  avg = np.mean(chunk)
  avg_rewards.append(np.mean(chunk))

plt.figure(figsize=(10,5))
plt.plot(range(0, len(reward_list), window), avg_rewards, marker='o',label="avg reward(50 ep)")
plt.xlabel("episode")
plt.ylabel("avg reward")
plt.grid(True)
plt.legend()
plt.show()
plt.close()